In [1]:
from flask import Flask, request, url_for, redirect, make_response, session
app = Flask(__name__)

# 1.1 Request 和 Response
这一章节内容涉及的名词和用法很绕很多，但按照操作逻辑进行整理时，一切都也会好理解不少。<br>
但别忘了先运行一下 ⬆ `Cell 1` ，该导的库和实例化得先整上。

## 请求 `Request`

想要研究 Request，首先得了解一下 Flask 对 Request 的理解—— `request` 对象。<br>

### Request 对象

在 Flask 中，`request` 对象是一个全局对象，用于访问客户端发送的 HTTP 请求数据。<br>
它包含了请求的所有信息，如请求方法、请求头、表单数据、文件上传、查询参数等。

### Request 方法

Request 方法旨在联系网页内容和URL。存在 GET 和 POST 两种。<br>

#### `GET`

GET 用于从 URL 传入参数。<br>
如 `localhost:5000/info?name=pylor&age=18` ，<br>
这里，在 `?` 后面的参数（也就是 `name` 和 `age` ），是 GET 参数。<br><br>
`GET` 参数使用 `args` 方法进行获取<br>

In [ ]:
# 代码 1.1
'''
字典中的内容是匹配自己输入的URL中的参数名的。
可以不严谨地理解为 URL 中定义的某变量，被 request.args[] 匹配读取下来，而读取的内容又可以变成 Python 的变量来进行操作。
'''
@app.route('/GETinfo')
def getinfo():
    name = request.args['name']
    age = request.args['age']
    return "姓名为{}，年龄为{}".format(name, age)

if __name__ == '__main__':
    app.run()

'''
然后，登陆 127.0.0.1:5000/GETinfo?name=pylor&age=3
就会返回相应的内容到网页上。
'''

当然，也可以没有参数，如 `127.0.0.1:5000/info` 也是 GET 请求。这里不再展示。<br><br>
为了让 `args` 的字典行为和 Python 接近，Flask 还提供了 `get` 方法来获取参数值。<br>
它们的区别仅在参数不存在时行为不同：<br>
- `args['nobody']` 会抛出 `KeyError` 错误；
- `args.get('nobody')` 则会返回 `None`。

`get` 方法还可以接受一个默认值参数，如 `args.get('name', 'default')`，当参数不存在时返回 `default` 。<br>

In [ ]:
# 代码 1.2
@app.route('/GETinfoget')
def getinfoget():
    name = request.args.get('nobodyname')                   #1
    age = request.args.get('nobodyage', 'OhNoAgeThere')     #1
    return "姓名为{}，年龄为{}".format(name, age)             #1
    # nobodydict = request.args['nobodydict']               #2
    # return "{}是不存在的".format(nobodydict)                #2

if __name__ == '__main__':
    app.run()

'''
登陆 127.0.0.1:5000/GETinfoget?name=pylor&age=3 试一试。
很显然，我们要查找的是 “nobodyxxx” ，但 URL 里写的是 xxx 。
前者会返回 “None” ，后者则会如预期一样，返回 “OhNoAgeThere”
也可以解除 7 、 8 行的注释，看看不存在的 URL 参数在 dict 上究竟会发生什么。
当然，也可以自行改写到 print() 的形式。文脉相通。
'''

#### `POST`

POST 请求用于向服务器发送数据，通常用于提交表单数据。<br>
在 Flask 中，可以通过在 route 装饰器中指定 `methods=['POST']` 来定义一个 POST 请求的路由。<br>
当然，也可以同时支持 GET 和 POST 请求，如 `methods=['GET', 'POST']`。<br>
POST 请求的数据可以通过 `request.form` 来获取，这个对象是一个字典，包含了表单提交的数据。<br>

代码如下：

In [ ]:
@app.route('/infopost', methods=['GET', 'POST'])
def infopost():
	if request.method == 'POST':
		name = request.form.get('name')
		return f"POST 参数 name={name}"
	else:
		name = request.args.get('name')
		return f"GET 参数 name={name}"

if __name__ == '__main__':
    app.run()
'''
可以登陆 localhost:5500/infopost?name=pylor 看看效果。
这里直接指定了 request.method 来强制它的类型是 POST。
实际上，文件尾还有一个更为直观的例子，可以看看。
'''

### URL 传参
除了刚刚我们涉及到的 `/path?key=value` 之外，URL 还能够通过路径本身来传递参数。<br>
如果只看 URL ，那可能会看到这种：<br><br>
`/path/user/pylor`<br><br>
这是网站中常见的个人信息页面路径结构，其中 `pylor` 就是一个参数。<br>
在 Flask 中，我们可以通过在 route 中使用尖括号 `< >` 来定义这样的参数。<br>
例如，定义一个 route `/user/<username>`，当访问 `/user/pylor` 时，`username = pylor` 。<br>

尖括号里的内容还可以指定类型，如 `<int:id>`<br>这样访问 `/user/123` 时，`id` 就会被转换成整数 `123` 。<br>
这种指定类型的前缀叫做 **“转换器”** 。其他数据类型也都有对应的转换器。<small>见 教材 P21 </small> <br>

详情见下：</a>

In [ ]:
# 代码 1.3
@app.route('/sharpquote/<name>')
def sharpquotenm(name):
    print(type(name))
    return "Hello, {}".format(name)
@app.route('/sharpquote/<int:id>')
def sharpquoteid(id):
    print(type(id))
    return "Hello, No. {}".format(id)

if __name__ == '__main__':
    app.run()

'''
登陆 127.0.0.1:5000/sharpquote/ThisIsAName 
或者 127.0.0.1:5000/sharpquote/1103
可以观察到网页显示出了我们的内容，并且终端也输出了对应的数据类型。
'''

### URL 反转：`url_for()`
神他妈 “URL反转”...谁的翻译？<br>
实际上，这玩意就是个从 route 名称反向生成 URL 的功能，和 “反转” 没半点关系，除非你说“此反转非彼反转”...那祝你开心。总之我不觉得这个翻译有多好。但毕竟我不是大佬，自个有数吧。<br>
`url_for` 做的事情就是根据 route 的**函数名**来对应 URL 。<br>
`url_for` 后面可以跟参数，这里的参数是 **route 函数所决定的动态部分**，例如 `id = 10`。<br>
- 要明白一点：Flask 中没有严格的顺序关系。<br>诚然，在 index 中使用了 `id = 10` 后再向下看才能看到 `id` 的定义确实会让人感到困扰，但这并不影响代码的运行。<br>

看代码<br>

In [ ]:
# 代码 1.4
@app.route('/')
def index():
    url = url_for('prod', id=10)
    return "URL反转内容{}".format(url)

@app.route('/prod/<id>')
def prod(id):
    return "您所需的产品编号是：{}".format(id)

if __name__ == '__main__':
    app.run()
'''
登陆 127.0.0.1:5000/ ，会注意到返回值有 “/prod/10” 字样，这代表着下面定义的变量与函数名已经被读取和生效；
登陆 127.0.0.1:5000/prod/13 ，或最后任意一个字符，会发现返回值仍然是对应值。
'''

### 跳转：`redirect()`
跳转是指在服务器端处理完请求后，告诉浏览器去访问另一个 URL 的过程。<br>
在 Flask 中，可以使用 `redirect` 函数来实现跳转。<br>
一般和 `url_for` 配合使用。<br>
正常情况下，可以直接写 URL：
```python
redirect('/user/pylor')
```
但 Flask 更推荐
```python
redirect(url_for('user', username='pylor'))
```
这类用法。

In [ ]:
@app.route('/')
def root():
    return redirect(url_for('user', username = 'pylor'))
@app.route('/user/<username>')
def user(username):
    return '已经跳转到 user，用户名为{}'.format(username)

if __name__ == '__main__':
    app.run()

### POST 请求后重定向 <small><small>POST/Redirect/GET, PRG 模式</small></small>
下面的代码就是一个典型的 POST 请求后重定向的完整例子。<br>
具体解释见代码内注释。<br>

对于 Session 管理，详见 [Response Cookie 和 Session](./Flask-Response.ipynb)

In [ ]:
app.secret_key = 'pylor520'
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        session['username'] = request.form['username']      # 若是 POST 请求，将用户名存入 Session ，
        return redirect(url_for('index'))                   # 然后重定向到 index
    return '''
        <form method="post">
            <p><input type=text name=username>
            <p><input type=submit value=登陆>
        </form>
    '''                                                     # 若否，写一个登陆表单 HTML。
#                                                             submit 按钮会建立一个 POST ，此时回到 是 ，逻辑闭环。
@app.route('/index')
def index():
    if 'username' in session:                               # 检查 Session 中是否存在 username ，即检查是否登陆
        return "你好，{}".format(session.get('username'))    # 若是，显示欢迎信息；
    return redirect(url_for('login'))                       # 若否，重定向到 login ，逻辑闭环。
        

if __name__ == '__main__':
    app.run()
'''
登陆 127.0.0.1:5000/login ，会注意到有一个提交文本框。输入文本之后，会被重定向到欢迎界面；
由于没有后端校验，按说 "username" not in session 的情况不会发生。
'''

## 响应 `response`
处理请求后返回给客户端的数据叫做响应。<br>
在 Flask 中，视图函数的返回值就是响应。<br>

响应的方法有三，见下。

In [ ]:
@app.route('/resdefault')
def resdefault():
    return "这个网页的 response 信息是 response 的默认返回属性。"
    # 这种情况下，等价于返回 Response("(上面返回文本)", status=200, mimetype="text/html")
    
@app.route('/resindex')
def resindex():
    return "这个网页的 response 信息是直接 return 三个参数得来的", 201, {'addheader':'response'}
@app.route('/resmake')

def resmake():
    res = make_response('这是通过make_response对象设置的参数', 202)
    res.headers["addheader"] = "responseSucceeeedddd!!!"
    return res

if __name__ == '__main__':
    app.run()
'''
分别登陆三个网页，这三个网页是 response 创建的三个方法：
直接 return ；
带参数 return ；
以及使用 make_response() 。
'''

### `jsonify()`
`jsonify` 是 Flask 提供的一个函数，用于将 Python 数据结构转换为 JSON 格式的响应。<br>
它会自动设置响应的 Content-Type 为 application/json，并且将数据转换为 JSON 格式字符串。<br>
使用 `jsonify` 可以方便地返回 JSON 数据给客户端，特别适合用于 API 开发。<br>